In [1]:
# Phase 3 Analysis Notebook: Advanced Feature Engineering
# ========================================================

# %% [markdown]
# # Phase 3: Advanced Feature Engineering Analysis
# 
# This notebook analyzes the engineered features created in Phase 3:
# - Event classification patterns
# - Entity extraction statistics
# - Feature correlations
# - Event-type performance analysis
# - Entity impact assessment

# %% Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# %% [markdown]
# ## 1. Load All Feature Datasets

# %% Load data
# Event classification results
events_df = pd.read_csv('data/processed/events_classified.csv')
print(f"Events classified: {len(events_df)} headlines")

# Entity extraction results
entities_df = pd.read_csv('data/processed/entities_extracted.csv')
print(f"Entities extracted: {len(entities_df)} headlines")

# Final model-ready dataset
final_df = pd.read_csv('data/final/model_ready_full.csv')
print(f"Final dataset: {len(final_df)} observations with {len(final_df.columns)} features")

# %% [markdown]
# ## 2. Event Classification Analysis

# %% Event distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
event_counts = events_df['event_type'].value_counts()
event_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Event Type Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Event Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.subplot(1, 2, 2)
event_confidence = events_df.groupby('event_type')['confidence'].mean().sort_values(ascending=False)
event_confidence.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Average Classification Confidence by Event', fontsize=14, fontweight='bold')
plt.xlabel('Event Type')
plt.ylabel('Average Confidence')
plt.xticks(rotation=45, ha='right')
plt.axhline(y=0.5, color='red', linestyle='--', label='50% threshold')
plt.legend()
plt.tight_layout()

plt.savefig('results/figures/event_analysis/event_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Event-type sentiment analysis
plt.figure(figsize=(12, 5))

# Box plot: Sentiment by event type
event_sentiment = events_df.groupby('event_type')['ensemble_sentiment'].apply(list)
plt.subplot(1, 2, 1)
events_df.boxplot(column='ensemble_sentiment', by='event_type', ax=plt.gca())
plt.title('Sentiment Distribution by Event Type', fontsize=14, fontweight='bold')
plt.xlabel('Event Type')
plt.ylabel('Ensemble Sentiment')
plt.xticks(rotation=45, ha='right')
plt.suptitle('')  # Remove default title

# Average sentiment by event
plt.subplot(1, 2, 2)
avg_sentiment = events_df.groupby('event_type')['ensemble_sentiment'].mean().sort_values()
colors = ['red' if x < 0 else 'green' for x in avg_sentiment]
avg_sentiment.plot(kind='barh', color=colors, edgecolor='black')
plt.title('Average Sentiment by Event Type', fontsize=14, fontweight='bold')
plt.xlabel('Average Ensemble Sentiment')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.tight_layout()

plt.savefig('results/figures/event_analysis/event_sentiment.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Event type by ticker
event_ticker_matrix = pd.crosstab(events_df['ticker'], events_df['event_type'])
plt.figure(figsize=(10, 6))
sns.heatmap(event_ticker_matrix, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Count'})
plt.title('Event Type Distribution by Ticker', fontsize=14, fontweight='bold')
plt.xlabel('Event Type')
plt.ylabel('Ticker')
plt.tight_layout()
plt.savefig('results/figures/event_analysis/event_ticker_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 3. Entity Extraction Analysis

# %% Entity mention statistics
entity_stats = pd.DataFrame({
    'CEO Mentions': [entities_df['mentions_ceo'].sum()],
    'Product Mentions': [entities_df['mentions_product'].sum()],
    'Competitor Mentions': [entities_df['mentions_competitor'].sum()],
    'With Numbers': [entities_df['has_numbers'].sum()],
    'With Percentage': [entities_df['has_percentage'].sum()]
})

plt.figure(figsize=(10, 5))
entity_stats.T.plot(kind='bar', legend=False, color='teal', edgecolor='black')
plt.title('Entity Extraction Summary', fontsize=14, fontweight='bold')
plt.xlabel('Entity Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('results/figures/event_analysis/entity_mentions.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Entity mention rates by ticker
entity_by_ticker = entities_df.groupby('ticker')[
    ['mentions_ceo', 'mentions_product', 'mentions_competitor']
].mean() * 100

plt.figure(figsize=(10, 6))
entity_by_ticker.plot(kind='bar', width=0.8, edgecolor='black')
plt.title('Entity Mention Rate by Ticker (%)', fontsize=14, fontweight='bold')
plt.xlabel('Ticker')
plt.ylabel('Mention Rate (%)')
plt.legend(['CEO', 'Product', 'Competitor'], loc='upper right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('results/figures/event_analysis/entity_by_ticker.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Entity sentiment impact
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# CEO sentiment impact
ceo_comparison = entities_df.groupby('mentions_ceo')['ensemble_sentiment'].mean()
axes[0].bar(['No CEO', 'CEO Mentioned'], ceo_comparison.values, 
            color=['lightcoral', 'lightgreen'], edgecolor='black')
axes[0].set_title('Sentiment: CEO Mentioned vs Not', fontweight='bold')
axes[0].set_ylabel('Average Sentiment')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Product sentiment impact
product_comparison = entities_df.groupby('mentions_product')['ensemble_sentiment'].mean()
axes[1].bar(['No Product', 'Product Mentioned'], product_comparison.values,
            color=['lightcoral', 'lightgreen'], edgecolor='black')
axes[1].set_title('Sentiment: Product Mentioned vs Not', fontweight='bold')
axes[1].set_ylabel('Average Sentiment')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Competitor sentiment impact
competitor_comparison = entities_df.groupby('mentions_competitor')['ensemble_sentiment'].mean()
axes[2].bar(['No Competitor', 'Competitor Mentioned'], competitor_comparison.values,
            color=['lightcoral', 'lightgreen'], edgecolor='black')
axes[2].set_title('Sentiment: Competitor Mentioned vs Not', fontweight='bold')
axes[2].set_ylabel('Average Sentiment')
axes[2].axhline(y=0, color='black', linestyle='-', linewidth=0.8)

plt.tight_layout()
plt.savefig('results/figures/event_analysis/entity_sentiment_impact.png', dpi=300, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 4. Feature Correlation Analysis

# %% Select key features for correlation
correlation_features = [
    'ensemble_sentiment_mean',
    'sentiment_variance_mean',
    'model_consensus_mean',
    'sentiment_earnings',
    'sentiment_product',
    'ceo_sentiment',
    'RSI',
    'MACD',
    'ATR',
    'volatility',
    'daily_return',
    'movement'
]

# Filter features that exist
available_corr_features = [f for f in correlation_features if f in final_df.columns]
corr_matrix = final_df[available_corr_features].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/event_analysis/feature_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Feature vs Target correlation
if 'movement' in final_df.columns:
    target_corr = final_df[available_corr_features].corrwith(final_df['movement']).sort_values(ascending=False)
    
    plt.figure(figsize=(10, 8))
    colors = ['green' if x > 0 else 'red' for x in target_corr.values]
    target_corr.plot(kind='barh', color=colors, edgecolor='black')
    plt.title('Feature Correlation with Target (Movement)', fontsize=14, fontweight='bold')
    plt.xlabel('Correlation Coefficient')
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    plt.tight_layout()
    plt.savefig('results/figures/event_analysis/target_correlation.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nTop 5 Features Correlated with Movement:")
    print(target_corr.head(5))

# %% [markdown]
# ## 5. Event-Type Predictive Power

# %% Event type vs movement
if 'movement' in final_df.columns and 'sentiment_earnings' in final_df.columns:
    # Calculate average movement by event sentiment bins
    event_types_cols = ['sentiment_earnings', 'sentiment_product', 'sentiment_analyst']
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for idx, event_col in enumerate(event_types_cols):
        if event_col in final_df.columns:
            # Bin sentiment into negative, neutral, positive
            final_df[f'{event_col}_bin'] = pd.cut(
                final_df[event_col], 
                bins=[-np.inf, -0.1, 0.1, np.inf],
                labels=['Negative', 'Neutral', 'Positive']
            )
            
            # Calculate movement rate by sentiment bin
            movement_by_sentiment = final_df.groupby(f'{event_col}_bin')['movement'].mean() * 100
            
            axes[idx].bar(movement_by_sentiment.index, movement_by_sentiment.values,
                         color=['red', 'gray', 'green'], edgecolor='black')
            axes[idx].set_title(f'{event_col.replace("sentiment_", "").title()} Event Impact',
                               fontweight='bold')
            axes[idx].set_ylabel('Movement UP Rate (%)')
            axes[idx].axhline(y=50, color='black', linestyle='--', label='Random (50%)')
            axes[idx].legend()
    
    plt.tight_layout()
    plt.savefig('results/figures/event_analysis/event_predictive_power.png', dpi=300, bbox_inches='tight')
    plt.show()

# %% [markdown]
# ## 6. Summary Statistics

# %% Final dataset summary
print("\n" + "="*60)
print("PHASE 3: FEATURE ENGINEERING SUMMARY")
print("="*60 + "\n")

print(f"Total Observations: {len(final_df)}")
print(f"Total Features: {len(final_df.columns) - 3}")  # Exclude date, ticker, target
print(f"Date Range: {final_df['date'].min()} to {final_df['date'].max()}")
print(f"Tickers: {final_df['ticker'].nunique()}")

print(f"\n📊 Feature Categories:")
print(f"  • Sentiment features: 12")
print(f"  • Event-specific features: 6")
print(f"  • Entity features: 5")
print(f"  • Technical indicators: 15")
print(f"  • Lagged features: 4")
print(f"  • Total: 42 features")

print(f"\n🎯 Target Distribution:")
if 'movement' in final_df.columns:
    movement_dist = final_df['movement'].value_counts()
    print(f"  • UP (1): {movement_dist.get(1, 0)} ({movement_dist.get(1, 0)/len(final_df)*100:.1f}%)")
    print(f"  • DOWN (0): {movement_dist.get(0, 0)} ({movement_dist.get(0, 0)/len(final_df)*100:.1f}%)")

print(f"\n📈 Event Classification:")
print(f"  • Total headlines classified: {len(events_df)}")
print(f"  • Average confidence: {events_df['confidence'].mean():.3f}")
print(f"  • Most common event: {events_df['event_type'].mode()[0]}")

print(f"\n🏷️ Entity Extraction:")
print(f"  • CEO mentions: {entities_df['mentions_ceo'].sum()} ({entities_df['mentions_ceo'].mean()*100:.1f}%)")
print(f"  • Product mentions: {entities_df['mentions_product'].sum()} ({entities_df['mentions_product'].mean()*100:.1f}%)")
print(f"  • Competitor mentions: {entities_df['mentions_competitor'].sum()} ({entities_df['mentions_competitor'].mean()*100:.1f}%)")

print("\n" + "="*60)
print("✅ PHASE 3 ANALYSIS COMPLETE!")
print("="*60)

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/events_classified.csv'

In [5]:


import pandas as pd

df = pd.read_csv(r"C:\Users\Asus\Downloads\financial-sentiment-nlp\data\final\model_ready_full.csv")
df['movement'].value_counts()



movement
1    314
0    288
Name: count, dtype: int64